# 06 · Root-cause walkthrough
End-to-end on 2-3 detected incidents: attribution + anomaly cluster + topology localisation.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60)
from networkanalysis.db.database import query_df, table_counts
from networkanalysis.pipeline.features import build_site_feature_table, KPI_DIRECTION, HEADLINE_KPIS

In [ ]:
rca = query_df("SELECT * FROM rca_finding WHERE scope='incident'")
det = query_df("SELECT * FROM incident_detected")
rca

In [ ]:
import json
for _, r in rca.iterrows():
    print("="*70)
    print(r.candidate_cause, "  conf", round(r.confidence,2), " matched:", r.matched_incident_id)
    print("entity:", r.candidate_entity)
    for e in json.loads(r.evidence): print("  -", e)
    print("ACTION:", r.recommended_action)

In [ ]:
# show the affected-site cluster for the first localised transport incident on the map
d = det[det.predicted_class=='transport'].iloc[0]
sites = json.loads(d.site_ids)
ll = query_df("SELECT site_id, lat, lon FROM dim_site")
plt.figure(figsize=(6,6))
plt.scatter(ll.lon, ll.lat, s=4, c="#ccc")
sub = ll[ll.site_id.isin(sites)]
plt.scatter(sub.lon, sub.lat, s=40, c="#e8663c")
plt.title(f"detected transport incident: {len(sites)} sites")